In [2]:
import os
import rasterio
from tensorflow.keras import layers, Model
from rasterio.windows import Window
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
input_data_dsm = '/home/ajai-krishna/work/GEO_AI/data/input/subsets/gujarath_dsm_RGBZ.tif'
input_data_dtm = '/home/ajai-krishna/work/GEO_AI/outputs/gujarath_dtm_RGBZ_filled.tif'
output_dtm_tif = '/home/ajai-krishna/work/GEO_AI/data/Training/dtm_patches'
output_dsm_tif = '/home/ajai-krishna/work/GEO_AI/data/Training/dsm_patches'
# output_trainin

In [ ]:
# def extract_patches(data_path, patch_size=256, stride=128, nodata_threshold=0.1, output_dir=None):

#     output_dir = Path(output_dir)
#     output_dir.mkdir(parents=True, exist_ok=True)

#     npy_dir = output_dir / "npy"
#     tif_dir = output_dir / "tif"

#     npy_dir.mkdir(exist_ok=True)
#     tif_dir.mkdir(exist_ok=True)

#     with rasterio.open(data_path) as src:

#         H = src.height
#         W = src.width
#         nodata_value = src.nodata
#         profile = src.profile.copy()

#         patch_id = 0
#         skipped = 0

#         for y in range(0, H - patch_size + 1, stride):
#             for x in range(0, W - patch_size + 1, stride):

#                 window = Window(x, y, patch_size, patch_size)

#                 patch = src.read(window=window).astype(np.float32)  # (bands,H,W)

#                 # nodata ratio check
#                 nodata_ratio = (patch == nodata_value).mean()

#                 if nodata_ratio > nodata_threshold:
#                     skipped += 1
#                     continue

#                 # ── Save NPY (for ML) ──
#                 patch_npy = np.transpose(patch, (1,2,0))  # (H,W,bands)

#                 np.save(npy_dir / f"patch_{patch_id}.npy", patch_npy)

#                 # ── Save TIF (georeferenced) ──
#                 transform = src.window_transform(window)

#                 patch_profile = profile.copy()
#                 patch_profile.update({
#                     "height": patch_size,
#                     "width": patch_size,
#                     "transform": transform
#                 })

#                 tif_path = tif_dir / f"patch_{patch_id}.tif"

#                 with rasterio.open(tif_path, "w", **patch_profile) as dst:
#                     dst.write(patch)

#                 patch_id += 1

#     print(f"Total patches  : {patch_id + skipped}")
#     print(f"Saved patches  : {patch_id}")
#     print(f"Skipped patches: {skipped}")

In [ ]:
# extract_patches(input_data_dsm,patch_size=256,stride=128,nodata_threshold=0.1,output_dir=output_dsm_tif)

In [ ]:
# extract_patches(input_data_dtm,patch_size=256,stride=128,nodata_threshold=0.1,output_dir=output_dtm_tif)

In [ ]:
def dsm_dtm_generator(
    dsm_path,
    dtm_path,
    patch_size=256,
    stride=128,
    batch_size=8
):

    with rasterio.open(dsm_path) as dsm_src, rasterio.open(dtm_path) as dtm_src:

        H, W = dsm_src.height, dsm_src.width
        nodata = dsm_src.nodata

        while True:

            X_batch, y_batch = [], []

            for y in range(0, H - patch_size + 1, stride):
                for x in range(0, W - patch_size + 1, stride):

                    window = Window(x, y, patch_size, patch_size)

                    dsm = dsm_src.read(window=window).astype(np.float32)
                    dtm = dtm_src.read(1, window=window).astype(np.float32)

                    # Skip bad patches
                    if (dsm == nodata).mean() > 0.1:
                        continue

                    # Input (RGB + DSM)
                    X = np.transpose(dsm[:4], (1, 2, 0))

                    # Target (DTM)
                    y_target = dtm[..., np.newaxis]

                    # Normalize
                    X[:, :, :3] /= 255.0          # RGB
                    X[:, :, 3] /= 100.0          # DSM height
                    y_target /= 100.0            # DTM height

                    X_batch.append(X)
                    y_batch.append(y_target)

                    if len(X_batch) == batch_size:
                        yield np.array(X_batch), np.array(y_batch)
                        X_batch, y_batch = [], []

In [ ]:
def build_unet(input_shape=(256,256,4)):

    inputs = layers.Input(input_shape)

    # Encoder
    c1 = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(64, 3, activation='relu', padding='same')(p1)
    p2 = layers.MaxPooling2D()(c2)

    # Bottleneck
    b = layers.Conv2D(128, 3, activation='relu', padding='same')(p2)

    # Decoder
    u1 = layers.UpSampling2D()(b)
    u1 = layers.concatenate([u1, c2])
    c3 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1)

    u2 = layers.UpSampling2D()(c3)
    u2 = layers.concatenate([u2, c1])
    c4 = layers.Conv2D(32, 3, activation='relu', padding='same')(u2)

    outputs = layers.Conv2D(1, 1, activation='linear')(c4)

    model = Model(inputs, outputs)

    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )

    return model

In [ ]:
model = build_unet()

gen = dsm_dtm_generator(
    dsm_path=input_data_dsm,
    dtm_path=input_data_dtm,
    patch_size=256,
    stride=128,
    batch_size=8
)

model.fit(
    gen,
    steps_per_epoch=1000,
    epochs=30
)

E0000 00:00:1773725091.674077    2090 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/30


I0000 00:00:1773725093.397270    2090 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
E0000 00:00:1773725095.621900    2090 util.cc:131] oneDNN supports DT_INT32 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


200/200 ━━━━━━━━━━━━━━━━━━━━ 388s 2s/step - loss: 16606.1777 - mae: 88.2186
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 349s 2s/step - loss: 19966.1914 - mae: 100.6265
Epoch 3/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 409s 2s/step - loss: 8546.5020 - mae: 53.3861
Epoch 4/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 384s 2s/step - loss: 19482.2598 - mae: 100.3343
Epoch 5/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 310s 2s/step - loss: 10529.3711 - mae: 63.1568
Epoch 6/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 297s 1s/step - loss: 7983.0850 - mae: 50.1342
Epoch 7/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 302s 2s/step - loss: 19082.5020 - mae: 98.5565
Epoch 8/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 316s 2s/step - loss: 6996.7295 - mae: 47.5924
Epoch 9/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 301s 2s/step - loss: 14293.8467 - mae: 77.8817
Epoch 10/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 291s 1s/step - loss: 11377.5225 - mae: 65.8124
Epoch 11/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 310s 2s/step - loss: 5775.5811 - mae: 40.3300
Epoch 12/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step 